<a href="https://colab.research.google.com/github/maminseo/maminseo/blob/main/%EB%B9%85%EB%8D%B0%EC%9D%B4%ED%84%B0%EB%B6%84%EC%84%9D_%EA%B3%BC%EC%A0%9C2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 데이터 엔지니어링 과제: Landing → Bronze Layer

**데이터 소스:** [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api)  
**실행 환경:** Google Colab / Python / PySpark  
**수집 대상:** 서울(위도 37.5665, 경도 126.9780), 2026-10-20 ~ 2026-10-22 (UTC 기준)

### 수행 목표
1. 외부 API 연결 및 원천 JSON 수집 → `landing/` 저장
2. Spark DataFrame API와 명시적 스키마를 사용하여 정제
3. 결측값 및 타입 오류 처리 → 날짜별 파티셔닝된 Parquet를 `bronze/`에 저장
4. 로그, 스키마, 데이터 샘플, 저장 결과 확인



## 0. 환경 준비 및 로깅

In [ ]:
%pip -q install pyspark==3.5.6 requests

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.6 which is incompatible.


In [ ]:
from pathlib import Path
import json, logging, sys
from datetime import datetime, timezone
import requests
from pyspark.sql import SparkSession, functions as F, types as T

BASE = Path('/content/weather_pipeline')
LANDING = BASE / 'landing'
BRONZE = BASE / 'bronze'
LOGS = BASE / 'logs'
for folder in (LANDING, BRONZE, LOGS):
    folder.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger('weather_pipeline')
logger.setLevel(logging.INFO)
logger.handlers.clear()
fmt = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
file_handler = logging.FileHandler(LOGS / 'pipeline.log', encoding='utf-8', mode='w')
file_handler.setFormatter(fmt)
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(fmt)
logger.addHandler(file_handler)
logger.addHandler(stream_handler)
logger.propagate = False

spark = (SparkSession.builder.appName('OpenMeteo_Landing_Bronze')
         .master('local[*]').getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
logger.info('Spark session ready: version=%s', spark.version)
print('Working directory:', BASE)

2026-10-31 10:09:33,074 | INFO | Spark session ready: version=3.5.6
Working directory: /content/weather_pipeline


## 1. 과제 01 — Open-Meteo API 연결 및 Landing 원본 수집

Archive API에서 **UTC 시간 기준 시간별 데이터**를 요청합니다. API 응답 전체를 가공하지 않고 JSON으로 저장해 Raw 데이터의 원형을 보존합니다.

In [ ]:
API_URL = 'https://archive-api.open-meteo.com/v1/archive'
PARAMS = {
    'latitude': 37.5665,
    'longitude': 126.9780,
    'start_date': '2026-10-20',
    'end_date': '2026-10-22',
    'hourly': 'temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m',
    'timezone': 'UTC'
}
logger.info('API request started: %s, period=%s ~ %s', API_URL, PARAMS['start_date'], PARAMS['end_date'])
try:
    response = requests.get(API_URL, params=PARAMS, timeout=45)
    response.raise_for_status()
    raw = response.json()
    if 'hourly' not in raw or 'time' not in raw['hourly']:
        raise ValueError(f'Unexpected API response: {str(raw)[:400]}')
    raw_path = LANDING / 'weather_raw.json'
    with raw_path.open('w', encoding='utf-8') as f:
        json.dump(raw, f, ensure_ascii=False, indent=2)
    logger.info('Landing JSON saved: %s (%d time rows)', raw_path, len(raw['hourly']['time']))
except Exception:
    logger.exception('API collection failed')
    raise

print('HTTP status:', response.status_code)
print('Saved file:', raw_path)
print('API timezone:', raw.get('timezone'))
print('Raw sample (first 3 timestamps):')
print(json.dumps({k: v[:3] if isinstance(v, list) else v
                  for k,v in raw['hourly'].items()}, ensure_ascii=False, indent=2))

2026-10-31 10:09:33,107 | INFO | API request started: https://archive-api.open-meteo.com/v1/archive, period=2026-10-20 ~ 2026-10-22
2026-10-31 10:09:33,193 | INFO | Landing JSON saved: /content/weather_pipeline/landing/weather_raw.json (72 time rows)
HTTP status: 200
Saved file: /content/weather_pipeline/landing/weather_raw.json
API timezone: GMT
Raw sample (first 3 timestamps):
{
  "time": [
    "2026-10-20T00:00",
    "2026-10-20T01:00",
    "2026-10-20T02:00"
  ],
  "temperature_2m": [
    13.8,
    16.0,
    17.6
  ],
  "relative_humidity_2m": [
    79,
    73,
    67
  ],
  "precipitation": [
    0.0,
    0.0,
    0.0
  ],
  "wind_speed_10m": [
    3.5,
    2.9,
    2.7
  ]
}


## 2. 과제 02 — 명시적 스키마를 이용한 Spark DataFrame 생성

Open-Meteo의 시간별 배열을 행 단위 레코드로 정렬합니다. `zip`을 사용해 시간과 측정값의 인덱스를 일치시킵니다. 수치 필드는 일단 문자열로 받아 Spark DataFrame API로 타입을 변환하고 검증합니다.

In [ ]:
with raw_path.open('r', encoding='utf-8') as f:
    landed = json.load(f)
hourly = landed['hourly']
fields = ['time', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m']
lengths = {k: len(hourly[k]) for k in fields}
if len(set(lengths.values())) != 1:
    raise ValueError(f'API arrays have inconsistent lengths: {lengths}')

records = [tuple(None if value is None else str(value) for value in vals)
           for vals in zip(*(hourly[k] for k in fields))]
raw_schema = T.StructType([
    T.StructField('time_raw', T.StringType(), True),
    T.StructField('temperature_raw', T.StringType(), True),
    T.StructField('humidity_raw', T.StringType(), True),
    T.StructField('precipitation_raw', T.StringType(), True),
    T.StructField('wind_speed_raw', T.StringType(), True)
])
df_raw = spark.createDataFrame(records, schema=raw_schema)
logger.info('Raw Spark DataFrame created: rows=%d', df_raw.count())
df_raw.printSchema()
df_raw.show(5, truncate=False)

2026-10-31 10:09:46,083 | INFO | Raw Spark DataFrame created: rows=72
root
 |-- time_raw: string (nullable = true)
 |-- temperature_raw: string (nullable = true)
 |-- humidity_raw: string (nullable = true)
 |-- precipitation_raw: string (nullable = true)
 |-- wind_speed_raw: string (nullable = true)

+----------------+---------------+------------+-----------------+--------------+
|time_raw        |temperature_raw|humidity_raw|precipitation_raw|wind_speed_raw|
+----------------+---------------+------------+-----------------+--------------+
|2026-10-20T00:00|13.8           |79          |0.0              |3.5           |
|2026-10-20T01:00|16.0           |73          |0.0              |2.9           |
|2026-10-20T02:00|17.6           |67          |0.0              |2.7           |
|2026-10-20T03:00|19.2           |60          |0.0              |1.9           |
|2026-10-20T04:00|20.3           |55          |0.0              |1.2           |
+----------------+---------------+------------+---

## 3. 타입 변환, 결측값 처리 및 검증

- 시간 문자열은 Timestamp로 변환합니다. API 요청 시간대가 UTC이므로 Spark 세션 시간대도 UTC로 고정합니다.
- 시간 누락/파싱 실패 행은 제거합니다.
- 수치 타입으로 변환할 수 없는 값은 `null`이 됩니다. 기온·습도·풍속의 결측은 제거하고, 강수량 결측은 0으로 대체합니다. **강수량 0 대체는 과제용 가정**이며, 실제 운영 환경에서는 결측을 0 강수로 간주할 수 있는지 데이터 정책 검토가 필요합니다.
- 습도 0~100%, 풍속·강수량 0 이상이라는 유효 범위를 확인합니다.
- 동일 시간 중복 레코드는 제거합니다.

In [ ]:
spark.conf.set('spark.sql.session.timeZone', 'UTC')
initial_count = df_raw.count()
df_typed = (df_raw
    .withColumn('observed_at', F.to_timestamp('time_raw', "yyyy-MM-dd'T'HH:mm"))
    .withColumn('temperature_c', F.col('temperature_raw').cast('double'))
    .withColumn('relative_humidity_pct', F.col('humidity_raw').cast('double'))
    .withColumn('precipitation_mm', F.col('precipitation_raw').cast('double'))
    .withColumn('wind_speed_kmh', F.col('wind_speed_raw').cast('double'))
    .select('observed_at', 'temperature_c', 'relative_humidity_pct',
            'precipitation_mm', 'wind_speed_kmh'))

print('Null counts BEFORE cleaning:')
df_typed.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df_typed.columns]).show()

clean = (df_typed
    .dropna(subset=['observed_at', 'temperature_c', 'relative_humidity_pct', 'wind_speed_kmh'])
    .fillna({'precipitation_mm': 0.0})
    .filter(F.col('relative_humidity_pct').between(0, 100))
    .filter((F.col('precipitation_mm') >= 0) & (F.col('wind_speed_kmh') >= 0))
    .dropDuplicates(['observed_at'])
    .withColumn('date', F.to_date('observed_at')))

final_count = clean.count()
logger.info('Cleaning completed: before=%d, after=%d, removed=%d',
            initial_count, final_count, initial_count-final_count)
if final_count == 0:
    raise ValueError('No valid records remain after cleaning')
expected_types = {'observed_at': T.TimestampType, 'temperature_c': T.DoubleType,
                  'relative_humidity_pct': T.DoubleType, 'precipitation_mm': T.DoubleType,
                  'wind_speed_kmh': T.DoubleType, 'date': T.DateType}
for field in clean.schema.fields:
    assert isinstance(field.dataType, expected_types[field.name]), f'Unexpected type: {field.name}'
logger.info('Explicit schema/type validation passed')
print('Bronze schema:')
clean.printSchema()
print('Cleaned sample:')
clean.orderBy('observed_at').show(10, truncate=False)
print('Null counts AFTER cleaning:')
clean.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in clean.columns]).show()
print('Rows by date:')
clean.groupBy('date').count().orderBy('date').show()

Null counts BEFORE cleaning:
+-----------+-------------+---------------------+----------------+--------------+
|observed_at|temperature_c|relative_humidity_pct|precipitation_mm|wind_speed_kmh|
+-----------+-------------+---------------------+----------------+--------------+
|          0|            0|                    0|               0|             0|
+-----------+-------------+---------------------+----------------+--------------+

2026-10-31 10:09:53,624 | INFO | Cleaning completed: before=72, after=72, removed=0
2026-10-31 10:09:53,634 | INFO | Explicit schema/type validation passed
Bronze schema:
root
 |-- observed_at: timestamp (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- relative_humidity_pct: double (nullable = true)
 |-- precipitation_mm: double (nullable = false)
 |-- wind_speed_kmh: double (nullable = true)
 |-- date: date (nullable = true)

Cleaned sample:
+-------------------+-------------+---------------------+----------------+--------------+-----

## 4. Bronze Layer — Parquet 저장 및 날짜별 파티셔닝

In [ ]:
bronze_path = str(BRONZE / 'weather_hourly')
(clean.write.mode('overwrite').partitionBy('date').parquet(bronze_path))
logger.info('Bronze Parquet written: %s, partitions=%d',
            bronze_path, clean.select('date').distinct().count())
print('Parquet output files:')
for p in sorted((BRONZE / 'weather_hourly').rglob('*')):
    if p.is_file() and (p.suffix == '.parquet' or p.name.startswith('_SUCCESS')):
        print(p.relative_to(BASE))

2026-10-31 10:10:01,173 | INFO | Bronze Parquet written: /content/weather_pipeline/bronze/weather_hourly, partitions=3
Parquet output files:
bronze/weather_hourly/_SUCCESS
bronze/weather_hourly/date=2026-10-20/part-00000-35b8e31a-096f-46ab-80f6-24c5693f4b6f.c000.snappy.parquet
bronze/weather_hourly/date=2026-10-21/part-00000-35b8e31a-096f-46ab-80f6-24c5693f4b6f.c000.snappy.parquet
bronze/weather_hourly/date=2026-10-22/part-00000-35b8e31a-096f-46ab-80f6-24c5693f4b6f.c000.snappy.parquet


In [ ]:
df_check = spark.read.parquet(bronze_path)
assert df_check.count() == final_count, 'Saved record count does not match!'
assert 'date' in df_check.columns, 'Partition column date missing!'
assert df_check.filter(F.col('date').isNull()).count() == 0
logger.info('Parquet reload verification passed: rows=%d', df_check.count())
print('Reloaded schema:')
df_check.printSchema()
print('Reloaded sample:')
df_check.orderBy('observed_at').show(5, truncate=False)
print('Saved rows by partition date:')
df_check.groupBy('date').count().orderBy('date').show()
print('Recent pipeline logs:')
print((LOGS / 'pipeline.log').read_text(encoding='utf-8')[-4000:])

2026-10-31 10:10:03,144 | INFO | Parquet reload verification passed: rows=72
Reloaded schema:
root
 |-- observed_at: timestamp (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- relative_humidity_pct: double (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- wind_speed_kmh: double (nullable = true)
 |-- date: date (nullable = true)

Reloaded sample:
+-------------------+-------------+---------------------+----------------+--------------+----------+
|observed_at        |temperature_c|relative_humidity_pct|precipitation_mm|wind_speed_kmh|date      |
+-------------------+-------------+---------------------+----------------+--------------+----------+
|2026-10-20 00:00:00|13.8         |79.0                 |0.0             |3.5           |2026-10-20|
|2026-10-20 01:00:00|16.0         |73.0                 |0.0             |2.9           |2026-10-20|
|2026-10-20 02:00:00|17.6         |67.0                 |0.0             |2.7           |2026-10-20|
|2026

In [ ]:
import shutil
archive = shutil.make_archive('/content/weather_pipeline_outputs', 'zip', root_dir=str(BASE))
print('Output archive:', archive)
print('Contains: landing JSON, bronze parquet, logs')
from google.colab import files
files.download(archive)

Output archive: /content/weather_pipeline_outputs.zip
Contains: landing JSON, bronze parquet, logs


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>